In [ ]:
import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import date
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint, uniform
from xgboost import XGBRegressor
print('imports ok')

In [ ]:
from pathlib import Path as _P
REPO_ROOT = _P.cwd()
if not (REPO_ROOT / 'data' / 'processed' / 'payment_delay_train.csv').exists():
    REPO_ROOT = _P.cwd().parent
TRAIN_CSV = REPO_ROOT / 'data' / 'processed' / 'payment_delay_train.csv'
VAL_CSV = REPO_ROOT / 'data' / 'processed' / 'payment_delay_validation.csv'
TEST_CSV = REPO_ROOT / 'data' / 'processed' / 'payment_delay_test.csv'
PIPE_BUNDLE = REPO_ROOT / 'models' / 'payment_delay_feature_pipeline.joblib'
DICT_CSV = REPO_ROOT / 'reports' / 'payment_delay_feature_dictionary.csv'
AUDIT_CSV = REPO_ROOT / 'reports' / 'payment_delay_leakage_audit.csv'
TARGET = 'delay_days'
ID_COLS = ['invoice_id', 'customer_id', 'invoice_date', 'due_date']
FORBIDDEN = ['payment_date', 'delay_days', 'is_late', 'invoice_status', 'invoice_id', 'customer_id']
RANDOM_SEED = 42
PLOT_DIR = REPO_ROOT / 'reports' / 'payment_delay'
PLOT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUT = REPO_ROOT / 'models' / 'payment_delay_regressor.joblib'
META_OUT = REPO_ROOT / 'models' / 'payment_delay_model_metadata.json'
COMP_OUT = REPO_ROOT / 'reports' / 'payment_delay_model_comparison.csv'
TUNE_OUT = REPO_ROOT / 'reports' / 'payment_delay_hyperparameter_results.csv'
TEST_PRED_OUT = REPO_ROOT / 'reports' / 'payment_delay_test_predictions.csv'
IMP_OUT = REPO_ROOT / 'reports' / 'payment_delay_feature_importance.csv'
CAT_OUT = REPO_ROOT / 'reports' / 'payment_delay_performance_by_category.csv'
AMT_OUT = REPO_ROOT / 'reports' / 'payment_delay_performance_by_amount.csv'
print('REPO_ROOT:', REPO_ROOT)

## 1. Project Objective

Regression on signed `delay_days` (early `<0` / on-time `=0` / late `>0`; negatives never clipped).
Baselines first, then Linear / RandomForest / XGBoost (+HistGradientBoosting), validation-MAE selection,
chronological tuning, once-only test evaluation, error/residual/category analysis, saved model + reports.
Test data is never used for selection or tuning.

## 2. Imports

Executed in the opening cells above: sklearn regressors/metrics, `RandomizedSearchCV`/`TimeSeriesSplit`, `XGBRegressor`, plotting. No classifier is used; the target stays continuous.

## 3. Configuration

Paths, target/identifier/forbidden lists, seed and output locations are configured in the cell above. Plots go to `reports/payment_delay/`.

## 4. Load Training/Validation/Test Data

Load the chronological feature datasets plus the engineering pipeline bundle, dictionary and audit (informational).

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)
print('train:', train_df.shape, '| val:', val_df.shape, '| test:', test_df.shape)
# Re-declare the history transformer under __main__ so the FE-notebook-pickled bundle can unpickle.
from sklearn.base import BaseEstimator, TransformerMixin
from typing import List, Optional
class LeakageSafeCustomerHistoryTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, customer_col='customer_id', date_col='invoice_date', delay_col='delay_days', amount_col='invoice_amount', invoice_id_col='invoice_id'):
        self.customer_col, self.date_col, self.delay_col, self.amount_col, self.invoice_id_col = customer_col, date_col, delay_col, amount_col, invoice_id_col
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        raise NotImplementedError('History already computed in feature datasets; bundle reused for provenance only.')
try:
    bundle = joblib.load(PIPE_BUNDLE)
    print('bundle keys:', list(bundle.keys()))
    print('bundle model features:', len(bundle.get('model_features', [])))
except Exception as e:
    bundle = None
    print('bundle load skipped (non-fatal, features already materialized):', type(e).__name__, str(e)[:200])
for p, n in [(DICT_CSV, 'dictionary'), (AUDIT_CSV, 'audit')]:
    print(n, 'exists:', p.exists())

## 5. Dataset Validation

Existence, shapes, column agreement, numeric dtypes, forbidden-column exclusion, NaN/inf checks.

In [ ]:
assert TRAIN_CSV.exists() and VAL_CSV.exists() and TEST_CSV.exists(), 'Missing input CSV'
assert list(train_df.columns) == list(val_df.columns) == list(test_df.columns), 'Column mismatch across splits'
print('Train shape:', train_df.shape)
print('Validation shape:', val_df.shape)
print('Test shape:', test_df.shape)
assert TARGET in train_df.columns, 'Target missing'
for nm, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    n_nan = int(d[TARGET].isna().sum())
    print(f'{nm} target NaN: {n_nan}')
    assert n_nan == 0, f'{nm} target has NaN'
for c in ['payment_date', 'is_late', 'invoice_status']:
    assert c not in train_df.columns, f'{c} must not be present'
for c in ['invoice_id', 'customer_id']:
    assert c in train_df.columns, f'{c} expected as identifier (not a feature)'
FEATURES = [c for c in train_df.columns if c not in ID_COLS + [TARGET]]
print('Number of features:', len(FEATURES))
assert train_df[FEATURES].isna().sum().sum() == 0, 'NaN in features'
assert np.isfinite(train_df[FEATURES].to_numpy(dtype=float)).all(), 'inf in train features'
assert np.isfinite(val_df[FEATURES].to_numpy(dtype=float)).all(), 'inf in val features'
assert np.isfinite(test_df[FEATURES].to_numpy(dtype=float)).all(), 'inf in test features'
print('dataset validation PASSED')

## 6. Target and Feature Validation

Signed target preserved; features identical across splits; categoricals already encoded (one-hot columns present); leakage re-checked against dictionary/audit.

In [ ]:
for nm, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"{nm}: mean={d[TARGET].mean():.3f} median={d[TARGET].median():.3f} std={d[TARGET].std():.3f} min={d[TARGET].min():.0f} max={d[TARGET].max():.0f} neg={(d[TARGET]<0).sum()} zero={(d[TARGET]==0).sum()} pos={(d[TARGET]>0).sum()}")
assert (train_df[TARGET] < 0).sum() > 0, 'Negatives were clipped - forbidden'
leak = set(FEATURES).intersection(FORBIDDEN)
print('forbidden in X:', leak if leak else 'none - PASS')
assert len(leak) == 0
ohe_cols = [c for c in FEATURES if c.startswith('cust_payment_terms_') or c.startswith('invoice_currency_')]
print('encoded categorical columns:', len(ohe_cols), ohe_cols)
assert len(ohe_cols) >= 3, 'Expected one-hot categoricals'
if DICT_CSV.exists():
    dd = pd.read_csv(DICT_CSV)
    print('dictionary rows:', len(dd))
if AUDIT_CSV.exists():
    ad = pd.read_csv(AUDIT_CSV)
    bad = ad[ad['Feature'].isin(FORBIDDEN) & (ad['Allowed?'] == 'YES')]
    print('audit violations:', len(bad), '(must be 0)')
    assert len(bad) == 0
X_train, y_train = train_df[FEATURES], train_df[TARGET].astype(float)
X_val, y_val = val_df[FEATURES], val_df[TARGET].astype(float)
X_test, y_test = test_df[FEATURES], test_df[TARGET].astype(float)
print('matrices:', X_train.shape, X_val.shape, X_test.shape)

## 7. Baseline Models

Mean-delay and median-delay baselines (constants from TRAIN only). Metrics: MAE / RMSE / R2 / MedAE. They set the bar ML must beat.

In [ ]:
def reg_metrics(y_true, y_pred):
    mse = float(mean_squared_error(y_true, y_pred))
    return {'MAE': float(mean_absolute_error(y_true, y_pred)), 'RMSE': float(np.sqrt(mse)),
            'R2': float(r2_score(y_true, y_pred)), 'MedAE': float(median_absolute_error(y_true, y_pred))}

RESULTS = {}
TIMES = {}
mean_c = float(y_train.mean())
med_c = float(y_train.median())
print(f'train mean={mean_c:.4f} median={med_c:.4f}')
for name, c in [('Mean Baseline', mean_c), ('Median Baseline', med_c)]:
    RESULTS[name] = {'train': reg_metrics(y_train, np.full_like(y_train, c, dtype=float)),
                     'val': reg_metrics(y_val, np.full_like(y_val, c, dtype=float))}
    TIMES[name] = 0.0
    print(name, {k: round(v, 4) for k, v in RESULTS[name]['val'].items()})

## 8. Linear Regression

Ordinary least squares `LinearRegression()`, fit on TRAIN only; train + validation predictions stored.

In [ ]:
t0 = time.time()
lr = LinearRegression()
lr.fit(X_train, y_train)
TIMES['Linear Regression'] = time.time() - t0
RESULTS['Linear Regression'] = {'train': reg_metrics(y_train, lr.predict(X_train)), 'val': reg_metrics(y_val, lr.predict(X_val))}
print('Linear Regression val:', {k: round(v, 4) for k, v in RESULTS['Linear Regression']['val'].items()}, f"({TIMES['Linear Regression']:.1f}s)")

## 9. Random Forest Regression

`RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)` — sensible baseline, no extensive pre-tuning.

In [ ]:
t0 = time.time()
rf = RandomForestRegressor(n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train, y_train)
TIMES['Random Forest'] = time.time() - t0
RESULTS['Random Forest'] = {'train': reg_metrics(y_train, rf.predict(X_train)), 'val': reg_metrics(y_val, rf.predict(X_val))}
print('Random Forest val:', {k: round(v, 4) for k, v in RESULTS['Random Forest']['val'].items()}, f"({TIMES['Random Forest']:.1f}s)")

## 10. XGBoost Regression

Main candidate: `XGBRegressor(objective='reg:squarederror', n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42)`. Plus optional `HistGradientBoostingRegressor` for comparison.

In [ ]:
t0 = time.time()
xgb = XGBRegressor(objective='reg:squarederror', n_estimators=500, learning_rate=0.05, max_depth=6,
                   subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_SEED, n_jobs=-1)
xgb.fit(X_train, y_train)
TIMES['XGBoost'] = time.time() - t0
RESULTS['XGBoost'] = {'train': reg_metrics(y_train, xgb.predict(X_train)), 'val': reg_metrics(y_val, xgb.predict(X_val))}
print('XGBoost val:', {k: round(v, 4) for k, v in RESULTS['XGBoost']['val'].items()}, f"({TIMES['XGBoost']:.1f}s)")
t0 = time.time()
hgb = HistGradientBoostingRegressor(random_state=RANDOM_SEED)
hgb.fit(X_train, y_train)
TIMES['HistGradientBoosting'] = time.time() - t0
RESULTS['HistGradientBoosting'] = {'train': reg_metrics(y_train, hgb.predict(X_train)), 'val': reg_metrics(y_val, hgb.predict(X_val))}
print('HistGradientBoosting val:', {k: round(v, 4) for k, v in RESULTS['HistGradientBoosting']['val'].items()}, f"({TIMES['HistGradientBoosting']:.1f}s)")

## 11. Initial Model Comparison

Train/val MAE-RMSE-R2-MedAE plus training time. Selection uses validation only (test untouched).

In [ ]:
comp = pd.DataFrame([{'Model': m, 'Train MAE': RESULTS[m]['train']['MAE'], 'Validation MAE': RESULTS[m]['val']['MAE'],
                      'Train RMSE': RESULTS[m]['train']['RMSE'], 'Validation RMSE': RESULTS[m]['val']['RMSE'],
                      'Train R2': RESULTS[m]['train']['R2'], 'Validation R2': RESULTS[m]['val']['R2'],
                      'Validation MedAE': RESULTS[m]['val']['MedAE'], 'Training Time': TIMES[m]} for m in RESULTS])
print(comp.sort_values('Validation MAE').round(4).to_string(index=False))

## 12. Hyperparameter Tuning

Tune the strongest ML candidate (XGBoost, preferred) with `RandomizedSearchCV` + `TimeSeriesSplit(n_splits=3)` (chronological — no shuffling, no KFold-shuffle, no test data). Small, hardware-aware budget.

In [ ]:
param_dist = {'n_estimators': randint(200, 800), 'learning_rate': uniform(0.02, 0.12), 'max_depth': randint(3, 9),
              'min_child_weight': randint(1, 8), 'subsample': uniform(0.6, 0.4), 'colsample_bytree': uniform(0.6, 0.4),
              'reg_alpha': uniform(0.0, 1.0), 'reg_lambda': uniform(0.5, 2.0)}
tscv = TimeSeriesSplit(n_splits=3)
search = RandomizedSearchCV(XGBRegressor(objective='reg:squarederror', random_state=RANDOM_SEED, n_jobs=-1),
                            param_distributions=param_dist, n_iter=12, scoring='neg_mean_absolute_error',
                            cv=tscv, random_state=RANDOM_SEED, n_jobs=-1, verbose=0)
t0 = time.time()
search.fit(X_train, y_train)
tune_secs = time.time() - t0
BEST_PARAMS = {k: (float(v) if isinstance(v, float) else int(v)) for k, v in search.best_params_.items()}
print('best val MAE (cv):', round(-float(search.best_score_), 4))
print('BEST_PARAMS:', BEST_PARAMS, f'({tune_secs:.1f}s)')
cv_res = pd.DataFrame(search.cv_results_)[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values('rank_test_score')
print(cv_res.head(5).to_string(index=False))

## 13. Tuned Model Evaluation

Refit XGBoost with best params on TRAIN; validation metrics; early-stopping attempt uses validation as eval_set (test never in eval_set; version-adaptive fallback to plain fit).

In [ ]:
xgb_tuned = XGBRegressor(objective='reg:squarederror', random_state=RANDOM_SEED, n_jobs=-1, **BEST_PARAMS)
t0 = time.time()
try:
    xgb_tuned.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    print('fit with validation eval_set (early-stopping-capable path)')
except TypeError as e:
    xgb_tuned.fit(X_train, y_train)
    print('plain fit fallback:', str(e)[:150])
TIMES['XGBoost-Tuned'] = time.time() - t0
RESULTS['XGBoost-Tuned'] = {'train': reg_metrics(y_train, xgb_tuned.predict(X_train)), 'val': reg_metrics(y_val, xgb_tuned.predict(X_val))}
print('XGBoost-Tuned val:', {k: round(v, 4) for k, v in RESULTS['XGBoost-Tuned']['val'].items()})

## 14. Final Model Selection

Selection on VALIDATION only. Primary: MAE. Secondary: RMSE / R2 / MedAE. R2 alone never decides. No unsupported claims.

In [ ]:
rank = sorted([m for m in RESULTS if 'Baseline' not in m], key=lambda m: RESULTS[m]['val']['MAE'])
print('validation MAE ranking:')
for m in rank:
    v = RESULTS[m]['val']
    print(f"  {m}: MAE={v['MAE']:.4f} RMSE={v['RMSE']:.4f} R2={v['R2']:.4f} MedAE={v['MedAE']:.4f}")
FINAL_NAME = rank[0]
print('FINAL MODEL:', FINAL_NAME)
print(f"beats mean baseline by {RESULTS['Mean Baseline']['val']['MAE'] - RESULTS[FINAL_NAME]['val']['MAE']:.4f} MAE days")

## 15. Final Test Evaluation

**Option B (documented):** after selection, combine TRAIN+VAL with the already-chosen hyperparameters, retrain, and evaluate ONCE on the untouched TEST set. Test was used for nothing else.

In [ ]:
FINAL_ESTIMATORS = {'Linear Regression': LinearRegression, 'Random Forest': RandomForestRegressor,
                    'XGBoost': XGBRegressor, 'XGBoost-Tuned': XGBRegressor, 'HistGradientBoosting': HistGradientBoostingRegressor}
FINAL_KWARGS = {'Linear Regression': {}, 'Random Forest': {'n_estimators': 300, 'random_state': RANDOM_SEED, 'n_jobs': -1},
                'XGBoost': {'objective': 'reg:squarederror', 'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 6,
                            'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': RANDOM_SEED, 'n_jobs': -1},
                'XGBoost-Tuned': dict({'objective': 'reg:squarederror', 'random_state': RANDOM_SEED, 'n_jobs': -1}, **BEST_PARAMS),
                'HistGradientBoosting': {'random_state': RANDOM_SEED}}
X_dev = pd.concat([X_train, X_val], ignore_index=True)
y_dev = pd.concat([y_train, y_val], ignore_index=True)
print(f'Option B refit on train+val: {X_dev.shape} (hyperparameters frozen from validation selection)')
final_model = FINAL_ESTIMATORS[FINAL_NAME](**FINAL_KWARGS[FINAL_NAME])
final_model.fit(X_dev, y_dev)
test_pred = final_model.predict(X_test)
assert np.isfinite(test_pred).all(), 'Non-finite test predictions'
TEST_METRICS = reg_metrics(y_test, test_pred)
print('FINAL TEST:', {k: round(v, 4) for k, v in TEST_METRICS.items()})
VAL_METRICS_FINAL = RESULTS[FINAL_NAME]['val']

## 16. Error Analysis

Per-invoice actual/predicted/residual/absolute-error table; bias, over/underprediction; breakdowns. Saved to `reports/payment_delay_test_predictions.csv`.

In [ ]:
err_df = pd.DataFrame({'invoice_id': test_df['invoice_id'].values, 'customer_id': test_df['customer_id'].values,
                       'actual_delay_days': y_test.values, 'predicted_delay_days': test_pred})
err_df['residual'] = err_df['actual_delay_days'] - err_df['predicted_delay_days']
err_df['absolute_error'] = err_df['residual'].abs()
err_df['direction'] = np.where(err_df['residual'] > 0, 'underprediction', np.where(err_df['residual'] < 0, 'overprediction', 'exact'))
print(f"mean residual={err_df['residual'].mean():.4f} median={err_df['residual'].median():.4f} MAE={err_df['absolute_error'].mean():.4f}")
print(err_df['direction'].value_counts().to_string())
for grp, m in [('early', err_df['actual_delay_days'] < 0), ('on-time', err_df['actual_delay_days'] == 0), ('late', err_df['actual_delay_days'] > 0)]:
    s = err_df.loc[m, 'absolute_error']
    print(f'{grp}: n={int(m.sum())} MAE={s.mean():.4f} MedAE={s.median():.4f}')
print('largest absolute errors:')
print(err_df.nlargest(5, 'absolute_error').round(2).to_string(index=False))
err_df.to_csv(TEST_PRED_OUT, index=False)
print('saved:', TEST_PRED_OUT, err_df.shape)

## 17. Residual Analysis

Actual-vs-predicted, residual-vs-predicted, residual/absolute-error distributions, delay over time. Saved under `reports/payment_delay/`. Large errors are analyzed, never deleted.

In [ ]:
def _save(name):
    p = PLOT_DIR / name
    plt.tight_layout(); plt.savefig(p, dpi=120); plt.show(); print('saved:', p)
plt.figure(figsize=(6,5)); plt.scatter(y_test, test_pred, s=4, alpha=0.3)
mn, mx = float(min(y_test.min(), test_pred.min())), float(max(y_test.max(), test_pred.max()))
plt.plot([mn, mx], [mn, mx], 'r--'); plt.xlabel('actual delay_days'); plt.ylabel('predicted delay_days')
plt.title(f'Actual vs Predicted (test, {FINAL_NAME})'); _save('09_actual_vs_predicted.png')
plt.figure(figsize=(6,5)); plt.scatter(test_pred, y_test.values - test_pred, s=4, alpha=0.3)
plt.axhline(0, color='r', ls='--'); plt.xlabel('predicted'); plt.ylabel('residual (actual-predicted)')
plt.title('Residual vs Predicted (test)'); _save('10_residual_vs_predicted.png')
plt.figure(figsize=(8,4)); plt.hist(y_test.values - test_pred, bins=60, edgecolor='black')
plt.title('Residual distribution (test)'); plt.xlabel('residual'); _save('11_residual_dist.png')
plt.figure(figsize=(8,4)); plt.hist(np.abs(y_test.values - test_pred), bins=60, edgecolor='black')
plt.title('Absolute error distribution (test)'); plt.xlabel('|error| days'); _save('12_abserror_dist.png')
tt = pd.DataFrame({'d': pd.to_datetime(test_df['invoice_date']), 'a': y_test.values, 'p': test_pred}).sort_values('d')
tt['ym'] = tt['d'].dt.to_period('M').astype(str)
g = tt.groupby('ym')[['a','p']].mean()
plt.figure(figsize=(10,4)); plt.plot(g.index, g['a'], marker='o', label='actual'); plt.plot(g.index, g['p'], marker='s', label='predicted')
plt.title('Actual vs predicted mean delay over time (test)'); plt.xticks(rotation=45); plt.legend(); _save('13_delay_over_time_test.png')
print('residual std:', round(float(np.std(y_test.values - test_pred)), 4))

## 18. Performance by Delay Category

Early (`<0`) / on-time (`==0`) / late (`>0`) on TEST. Overall MAE can hide late-invoice weakness. Saved to CSV.

In [ ]:
rows = []
for name, m in [('Early', y_test < 0), ('On-time', y_test == 0), ('Late', y_test > 0)]:
    yt, yp = y_test[m], test_pred[m.values]
    mse = float(mean_squared_error(yt, yp))
    rows.append({'category': name, 'MAE': float(mean_absolute_error(yt, yp)), 'RMSE': float(np.sqrt(mse)),
                 'MedAE': float(median_absolute_error(yt, yp)), 'n': int(m.sum())})
cat_df = pd.DataFrame(rows)
print(cat_df.round(4).to_string(index=False))
cat_df.to_csv(CAT_OUT, index=False)
print('saved:', CAT_OUT)

## 19. Performance by Customer History

TEST split by `has_payment_history` (>=0.5 vs <0.5): new vs returning customers.

In [ ]:
hist_flag = (test_df['has_payment_history'].astype(float) >= 0.5).values
rows = []
for name, m in [('with_history', hist_flag), ('without_history', ~hist_flag)]:
    yt, yp = y_test[m], test_pred[m]
    mse = float(mean_squared_error(yt, yp))
    rows.append({'group': name, 'MAE': float(mean_absolute_error(yt, yp)), 'RMSE': float(np.sqrt(mse)),
                 'R2': float(r2_score(yt, yp)) if (m.sum() > 2 and yt.nunique() > 1) else float('nan'),
                 'MedAE': float(median_absolute_error(yt, yp)), 'n': int(m.sum())})
hist_df = pd.DataFrame(rows)
print(hist_df.round(4).to_string(index=False))

## 20. Performance by Invoice Amount

TEST quantile buckets Low/Medium/High on `invoice_amount`. Saved to CSV.

In [ ]:
q33, q66 = float(test_df['invoice_amount'].quantile(0.33)), float(test_df['invoice_amount'].quantile(0.66))
print(f'quantile bounds: Low<={q33:.2f} Medium<={q66:.2f} High>')
buck = pd.cut(test_df['invoice_amount'], [-np.inf, q33, q66, np.inf], labels=['Low', 'Medium', 'High'])
rows = []
for b in ['Low', 'Medium', 'High']:
    m = (buck == b).values
    yt, yp = y_test[m], test_pred[m]
    mse = float(mean_squared_error(yt, yp))
    rows.append({'bucket': b, 'MAE': float(mean_absolute_error(yt, yp)), 'RMSE': float(np.sqrt(mse)),
                 'MedAE': float(median_absolute_error(yt, yp)), 'n': int(m.sum())})
amt_df = pd.DataFrame(rows)
print(amt_df.round(4).to_string(index=False))
amt_df.to_csv(AMT_OUT, index=False)
print('saved:', AMT_OUT)

## 21. Feature Importance

Native importance of the final tree model (+RF reference). CSV with `feature, importance, rank` and horizontal plot. Importance is associative, not causal.

In [ ]:
def _native_importance(model, Xt=X_val, yt=y_val):
    if hasattr(model, 'feature_importances_'):
        return np.asarray(model.feature_importances_, dtype=float), 'native'
    if hasattr(model, 'coef_'):
        return np.abs(np.asarray(model.coef_, dtype=float)).ravel(), 'abs-coef'
    from sklearn.inspection import permutation_importance
    r = permutation_importance(model, Xt, yt, n_repeats=5, random_state=RANDOM_SEED, n_jobs=-1, scoring='neg_mean_absolute_error')
    return np.asarray(r.importances_mean, dtype=float), 'permutation-mae'
imp, imp_kind = _native_importance(final_model)
print('importance kind:', imp_kind)
imp_df = pd.DataFrame({'feature': FEATURES, 'importance': imp})
imp_df['rank'] = imp_df['importance'].rank(ascending=False, method='min').astype(int)
imp_df = imp_df.sort_values('importance', ascending=False).reset_index(drop=True)
print(imp_df.head(12).round(4).to_string(index=False))
imp_df.to_csv(IMP_OUT, index=False)
print('saved:', IMP_OUT)
plt.figure(figsize=(8, 6)); top = imp_df.head(15).iloc[::-1]
plt.barh(top['feature'], top['importance']); plt.title(f'Feature importance ({imp_kind}, test-final {FINAL_NAME}, top 15)')
plt.tight_layout(); plt.savefig(PLOT_DIR / '14_feature_importance.png', dpi=120); plt.show()
print('saved:', PLOT_DIR / '14_feature_importance.png')

## 22. SHAP Explainability

If the final model is XGBoost and SHAP is installed: SHAP importance, summary and dependence plots under `reports/payment_delay/`. Otherwise skipped with a logged reason.

In [ ]:
SHAP_DONE = False
if FINAL_NAME.startswith('XGBoost'):
    try:
        import shap
        expl = shap.TreeExplainer(final_model)
        Xs = X_test.iloc[:1000]
        sv = expl.shap_values(Xs)
        print('SHAP values:', np.shape(sv))
        plt.figure(); shap.summary_plot(sv, Xs, show=False, max_display=15)
        plt.tight_layout(); plt.savefig(PLOT_DIR / '15_shap_summary.png', dpi=120, bbox_inches='tight'); plt.show()
        print('saved:', PLOT_DIR / '15_shap_summary.png')
        plt.figure(); shap.summary_plot(sv, Xs, plot_type='bar', show=False, max_display=15)
        plt.tight_layout(); plt.savefig(PLOT_DIR / '16_shap_bar.png', dpi=120, bbox_inches='tight'); plt.show()
        print('saved:', PLOT_DIR / '16_shap_bar.png')
        topf = imp_df.iloc[0]['feature']
        plt.figure()
        try:
            shap.dependence_plot(topf, sv, Xs, show=False)
        except Exception:
            plt.scatter(Xs[topf], sv[:, list(Xs.columns).index(topf)], s=4, alpha=0.4)
            plt.xlabel(topf); plt.ylabel('SHAP value')
        plt.tight_layout(); plt.savefig(PLOT_DIR / '17_shap_dependence.png', dpi=120, bbox_inches='tight'); plt.show()
        print('saved:', PLOT_DIR / '17_shap_dependence.png', f'(feature: {topf})')
        SHAP_DONE = True
    except Exception as e:
        print('SHAP skipped:', type(e).__name__, str(e)[:300])
else:
    print(f'SHAP skipped: final model is {FINAL_NAME}, SHAP analysis specified for XGBoost final.')
print('SHAP_DONE:', SHAP_DONE)

## 23. Final Prediction Analysis

Sanity checks (finite, no NaN/inf) and actual-vs-predicted distribution comparison. Predictions are never clipped without a documented business reason.

In [ ]:
print('finite:', bool(np.isfinite(test_pred).all()), '| NaN:', int(pd.isna(test_pred).sum()))
print(f"min={float(test_pred.min()):.2f} max={float(test_pred.max()):.2f} mean={float(test_pred.mean()):.2f} median={float(np.median(test_pred)):.2f}")
print(f"actual: min={float(y_test.min()):.0f} max={float(y_test.max()):.0f} mean={float(y_test.mean()):.2f} median={float(y_test.median()):.2f}")
plt.figure(figsize=(8,4))
plt.hist(y_test, bins=60, alpha=0.5, label='actual', edgecolor='black')
plt.hist(test_pred, bins=60, alpha=0.5, label='predicted', edgecolor='black')
plt.title('Actual vs predicted distribution (test)'); plt.xlabel('delay_days'); plt.legend()
plt.tight_layout(); plt.savefig(PLOT_DIR / '18_pred_distribution.png', dpi=120); plt.show()
print('saved:', PLOT_DIR / '18_pred_distribution.png')

## 24. Save Final Model

Persist `models/payment_delay_regressor.joblib` (estimator + feature list + helpers) and metadata JSON.

In [ ]:
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({'model': final_model, 'model_name': FINAL_NAME, 'features': FEATURES, 'target': TARGET}, MODEL_OUT)
print('saved:', MODEL_OUT)
FINAL_HP = FINAL_KWARGS[FINAL_NAME]
metadata = {'model_name': f'payment_delay_{FINAL_NAME.lower().replace(" ", "_").replace("-", "_")}',
            'model_type': FINAL_NAME, 'target': TARGET, 'training_rows': int(len(X_dev)),
            'validation_rows': int(len(X_val)), 'test_rows': int(len(X_test)), 'number_of_features': len(FEATURES),
            'training_date': str(date.today()), 'hyperparameters': {k: (float(v) if isinstance(v, (np.floating,)) else v) for k, v in FINAL_HP.items()},
            'validation_metrics': {k: round(float(v), 4) for k, v in VAL_METRICS_FINAL.items()},
            'test_metrics': {k: round(float(v), 4) for k, v in TEST_METRICS.items()},
            'feature_list': FEATURES, 'random_seed': RANDOM_SEED,
            'selection': 'Option B refit on train+val; hyperparameters frozen from validation; test evaluated once'}
META_OUT.write_text(json.dumps(metadata, indent=2))
print('saved:', META_OUT)
print(json.dumps(metadata, indent=2))

## 25. Save Metrics and Reports

`reports/payment_delay_model_comparison.csv` (model, MAE, RMSE, R2, MedAE, training_time_seconds) and hyperparameter results.

In [ ]:
mcomp = pd.DataFrame([{'model': m, 'MAE': RESULTS[m]['val']['MAE'], 'RMSE': RESULTS[m]['val']['RMSE'],
                       'R2': RESULTS[m]['val']['R2'], 'MedAE': RESULTS[m]['val']['MedAE'],
                       'training_time_seconds': TIMES.get(m, 0.0)} for m in RESULTS])
mcomp.to_csv(COMP_OUT, index=False)
print('saved:', COMP_OUT); print(mcomp.round(4).to_string(index=False))
cv_out = cv_res.copy(); cv_out['params'] = cv_out['params'].astype(str)
cv_out.to_csv(TUNE_OUT, index=False)
print('saved:', TUNE_OUT, cv_out.shape)

## 26. Reusable Inference Function

`predict_payment_delay(invoice_data)`: same feature order, same trained estimator, returns predicted `delay_days`. Raw-to-features must flow through the saved FE pipeline first; this function reuses the exact trained estimator without re-creating preprocessing.

In [ ]:
def predict_payment_delay(invoice_data):
    """Predict delay_days for model-ready rows (same FEATURES). Returns np.ndarray of floats."""
    obj = joblib.load(MODEL_OUT)
    mdl, feats = obj['model'], obj['features']
    X = invoice_data[feats] if isinstance(invoice_data, pd.DataFrame) else pd.DataFrame(invoice_data, columns=feats)
    pred = np.asarray(mdl.predict(X), dtype=float)
    assert np.isfinite(pred).all(), 'Non-finite predictions'
    return pred
demo = predict_payment_delay(X_test.iloc[:5])
print('demo predictions (first 5 test rows):', np.round(demo, 2).tolist())
print('actual:', y_test.iloc[:5].tolist())

## 27. Final Summary

Counts, baseline vs best validation, once-only test results, model, parameters and artifact paths.

In [ ]:
print('========================================')
print('FLOWCRED AI PAYMENT DELAY REGRESSION')
print('========================================')
print(f'Training samples: {len(X_train)}')
print(f'Validation samples: {len(X_val)}')
print(f'Test samples: {len(X_test)}')
print(f'Number of features: {len(FEATURES)}')
print(f"Baseline MAE: {RESULTS['Mean Baseline']['val']['MAE']:.4f} (mean) / {RESULTS['Median Baseline']['val']['MAE']:.4f} (median)")
print(f"Best model validation MAE: {VAL_METRICS_FINAL['MAE']:.4f}")
print(f"Best model validation RMSE: {VAL_METRICS_FINAL['RMSE']:.4f}")
print(f"Best model validation R2: {VAL_METRICS_FINAL['R2']:.4f}")
print('FINAL TEST RESULTS')
print('------------------')
print(f"Test MAE: {TEST_METRICS['MAE']:.4f}")
print(f"Test RMSE: {TEST_METRICS['RMSE']:.4f}")
print(f"Test R2: {TEST_METRICS['R2']:.4f}")
print(f"Test Median Absolute Error: {TEST_METRICS['MedAE']:.4f}")
print(f'Final model: {FINAL_NAME}')
print(f'Model parameters: {FINAL_KWARGS[FINAL_NAME]}')
print('========================================')
for p in [MODEL_OUT, META_OUT, COMP_OUT, TUNE_OUT, TEST_PRED_OUT, IMP_OUT, CAT_OUT, AMT_OUT]:
    print(('OK ' if p.exists() else 'MISSING ') + str(p))
print('plots dir:', PLOT_DIR)
print('========================================')
print('Business read: -5 ~= 5 days early; 0 ~= on due date; +12 ~= 12 days late. No binary conversion here.')